# Phase 5: Characterize → Merge → Optimize

Library feedback loop using the Python facade.

**Prerequisites:** `uv sync` and `uv run maturin develop -m crates/py/Cargo.toml`

In [ ]:
import rflux

print("rflux version:", rflux.__version__)
print("core version:", rflux.core_version())

## 1. Characterize compound cell

In [ ]:
char_circuit = rflux.Circuit("compound")
source = char_circuit.add_node("port", "source")
gate = char_circuit.add_node("cell", "gate")
sink = char_circuit.add_node("port", "sink")
char_circuit.connect(source, 0, gate, 0)
char_circuit.connect(gate, 0, sink, 0)

char_report = rflux.characterize_compound_cell(char_circuit, cell_name="macro_buf")
char_report

## 2. Consumer timing with characterized library

In [ ]:
consumer = rflux.Circuit("consumer")
consumer_source = consumer.add_node("port", "consumer_source")
macro_buf = consumer.add_node("macro", "macro_buf")
consumer_sink = consumer.add_node("dff", "consumer_sink")
consumer.connect(consumer_source, 0, macro_buf, 0)
consumer.connect(macro_buf, 0, consumer_sink, 0)

timing = rflux.analyze_timing(
    consumer,
    characterized_library_entries=[char_report.generated_library_json],
)
timing

## 3. Library-aware design optimization

In [ ]:
design = rflux.optimize_design_with_characterized_library(
    consumer,
    [char_report.generated_library_json],
)
print("design score:", design.design_optimization_score)
print("optimized pessimistic setup slack (ps):", design.optimized_statistical.worst_pessimistic_setup_slack_ps)
design